<a href="https://colab.research.google.com/github/fhriads/Komputasi-Numerik/blob/main/01_Sistem_Persamaan_Linear_3_Variabel/SPL3V_Eliminasi_Gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SISTEM PERSAMAAN LINEAR 3 VARIABEL (SPL 3V)
## Implementasi Komputasi Numerik: Eliminasi Gauss dengan Partial Pivoting & Back Substitution

- **Mata Kuliah** : Komputasi Numerik
- **Penyusun**    : Fahri Adis Al Hafni
- **Institusi**   : Universitas Dr. Soetomo (Unitomo) Surabaya

---
### Deskripsi Masalah
Menyelesaikan sistem persamaan linear tiga variabel:
$$
\begin{aligned}
a_{11}x_1 + a_{12}x_2 + a_{13}x_3 &= b_1 \\
a_{21}x_1 + a_{22}x_2 + a_{23}x_3 &= b_2 \\
a_{31}x_1 + a_{32}x_2 + a_{33}x_3 &= b_3
\end{aligned}
$$
Representasi matriks: $A \mathbf{x} = \mathbf{b}$.

### Cell 1: Import Library Pendukung
Pada cell ini kita mengimpor pustaka `numpy` untuk manipulasi array numerik berkecepatan tinggi dan presisi ganda (*float64*).

In [ ]:
# Mengimpor pustaka NumPy untuk operasi matriks dan komputasi numerik vektorisasi
import numpy as np

# Mengatur format cetak array floating-point agar rapi dengan 6 digit di belakang koma
np.set_printoptions(precision=6, suppress=True)

# Menampilkan pesan konfirmasi bahwa pustaka berhasil dimuat
print("Library NumPy berhasil diimpor dengan versi:", np.__version__)

Library NumPy berhasil diimpor dengan versi: 2.4.6


### Cell 2: Definisi Matriks Koefisien (A) dan Vektor Konstanta (b)
Kita definisikan sistem persamaan linear yang akan diuji:
$$
\begin{aligned}
4x_1 + x_2 - 2x_3 &= 9 \\
2x_1 + 5x_2 + x_3 &= 19 \\
x_1 - 2x_2 + 6x_3 &= 5
\end{aligned}
$$

In [ ]:
# Mendefinisikan matriks koefisien A berukuran 3x3 dengan tipe data float64
A = np.array([
    [4.0,  1.0, -2.0],
    [2.0,  5.0,  1.0],
    [1.0, -2.0,  6.0]
], dtype=np.float64)

# Mendefinisikan vektor konstanta b berukuran 3 elemen (ruas kanan persamaan)
b = np.array([9.0, 19.0, 5.0], dtype=np.float64)

# Menampilkan dimensi dan nilai matriks A
print("=== Matriks Koefisien A (Ukuran: {}x{}) ===".format(A.shape[0], A.shape[1]))
print(A)

# Menampilkan nilai vektor konstanta b
print("\n=== Vektor Konstanta b ===")
print(b)

=== Matriks Koefisien A (Ukuran: 3x3) ===
[[ 4.  1. -2.]
 [ 2.  5.  1.]
 [ 1. -2.  6.]]

=== Vektor Konstanta b ===
[ 9. 19.  5.]


### Cell 3: Pembentukan Matriks Teraugmentasi [A | b]
Menggabungkan matriks $A$ dan vektor $b$ menjadi satu matriks teraugmentasi berukuran $3 \times 4$ untuk memudahkan Operasi Baris Elementer (OBE).

In [ ]:
# Mengubah dimensi vektor b menjadi matriks kolom 3x1 menggunakan reshape
b_kolom = b.reshape(-1, 1)

# Menggabungkan matriks A dan kolom b secara horizontal (axis=1) untuk membentuk matriks augmented [A|b]
augmented_matrix = np.hstack([A, b_kolom])

# Menampilkan matriks teraugmentasi sebelum eliminasi
print("=== Matriks Teraugmentasi Awal [A | b] ===")
for i in range(len(augmented_matrix)):
    # Menampilkan 3 elemen koefisien dan 1 elemen konstanta terpisah dengan tanda pipet '|'
    koef = " ".join(f"{augmented_matrix[i, j]:10.4f}" for j in range(3))
    konst = f"{augmented_matrix[i, 3]:10.4f}"
    print(f"Baris {i+1} : [ {koef} | {konst} ]")

=== Matriks Teraugmentasi Awal [A | b] ===
Baris 1 : [     4.0000     1.0000    -2.0000 |     9.0000 ]
Baris 2 : [     2.0000     5.0000     1.0000 |    19.0000 ]
Baris 3 : [     1.0000    -2.0000     6.0000 |     5.0000 ]


### Cell 4: Eliminasi Maju (Forward Elimination) dengan Partial Pivoting
Pada cell ini dilakukan transformasi matriks teraugmentasi menjadi matriks segitiga atas.
- Dilengkapi **pivoting parsial** (mencari nilai pivot dengan nilai mutlak terbesar pada kolom terkait) untuk mencegah *division by zero* dan meminimalkan perambatan galat pembulatan (*round-off error*).

In [ ]:
# Mendapatkan jumlah baris (jumlah variabel n = 3)
n = len(b)

# Membuat salinan matriks augmented agar data asli tidak termodifikasi
M = augmented_matrix.copy()

print("=== PROSES ELIMINASI MAJU (FORWARD ELIMINATION) ===\n")

# Iterasi untuk setiap kolom pivot k dari 0 sampai n-2
for k in range(n - 1):
    print(f"--- Tahap Pivot Kolom {k + 1} ---")

    # 1. PIVOTING PARSIAL: Cari baris p (p >= k) dengan nilai mutlak terbesar pada kolom k
    baris_pivot = k + np.argmax(np.abs(M[k:, k]))
    nilai_pivot_terbesar = np.abs(M[baris_pivot, k])

    # Cek apakah nilai pivot terlalu kecil mendekati nol (singularitas matriks)
    if nilai_pivot_terbesar < 1e-12:
        raise ValueError(f"Matriks singular atau mendekati singular pada kolom {k+1}. SPL tidak memiliki solusi unik.")

    # Lakukan pertukaran baris jika baris pivot tidak berada di baris k saat ini
    if baris_pivot != k:
        print(f"-> Tukar Baris {k + 1} dengan Baris {baris_pivot + 1} (Pivot = {M[baris_pivot, k]:.4f})")
        M[[k, baris_pivot]] = M[[baris_pivot, k]]
    else:
        print(f"-> Baris {k + 1} sudah optimal sebagai pivot (Nilai Pivot = {M[k, k]:.4f})")

    # 2. ELIMINASI ELEMEN DI BAWAH PIVOT: Jadikan nol elemen M[i, k] untuk i > k
    for i in range(k + 1, n):
        # Hitung faktor pengali m_ik = M[i, k] / M[k, k]
        faktor = M[i, k] / M[k, k]
        print(f"   Baris {i + 1} := Baris {i + 1} - ({faktor:.6f}) * Baris {k + 1}")

        # Terapkan Operasi Baris Elementer ke seluruh elemen pada baris i
        M[i, k:] = M[i, k:] - faktor * M[k, k:]

    # Tampilkan matriks setelah tahap eliminasi kolom k selesai
    print("\nKondisi Matriks Teraugmentasi Saat Ini:")
    for r in range(n):
        koef = " ".join(f"{M[r, c]:10.4f}" for c in range(n))
        konst = f"{M[r, n]:10.4f}"
        print(f"[ {koef} | {konst} ]")
    print()

# Validasi elemen pivot diagonal terakhir
if np.abs(M[n - 1, n - 1]) < 1e-12:
    raise ValueError("Matriks singular pada elemen diagonal terakhir M[3, 3].")

print("Proses Eliminasi Maju Selesai. Terbentuk Matriks Segitiga Atas.")

=== PROSES ELIMINASI MAJU (FORWARD ELIMINATION) ===

--- Tahap Pivot Kolom 1 ---
-> Baris 1 sudah optimal sebagai pivot (Nilai Pivot = 4.0000)
   Baris 2 := Baris 2 - (0.500000) * Baris 1
   Baris 3 := Baris 3 - (0.250000) * Baris 1

Kondisi Matriks Teraugmentasi Saat Ini:
[     4.0000     1.0000    -2.0000 |     9.0000 ]
[     0.0000     4.5000     2.0000 |    14.5000 ]
[     0.0000    -2.2500     6.5000 |     2.7500 ]

--- Tahap Pivot Kolom 2 ---
-> Baris 2 sudah optimal sebagai pivot (Nilai Pivot = 4.5000)
   Baris 3 := Baris 3 - (-0.500000) * Baris 2

Kondisi Matriks Teraugmentasi Saat Ini:
[     4.0000     1.0000    -2.0000 |     9.0000 ]
[     0.0000     4.5000     2.0000 |    14.5000 ]
[     0.0000     0.0000     7.5000 |    10.0000 ]

Proses Eliminasi Maju Selesai. Terbentuk Matriks Segitiga Atas.


### Cell 5: Substitusi Mundur (Back Substitution)
Setelah matriks berbentuk segitiga atas, kita menghitung nilai variabel dari yang terbawah ($x_3$) bergerak mundur ke atas ($x_2$ lalu $x_1$).
Rumus substitusi mundur:
$$x_i = \frac{b'_i - \sum_{j=i+1}^n a'_{ij} x_j}{a'_{ii}}$$

In [ ]:
# Inisialisasi vektor nol untuk menampung 3 nilai solusi x
x = np.zeros(n, dtype=np.float64)

print("=== PROSES SUBSTITUSI MUNDUR (BACK SUBSTITUTION) ===\n")

# 1. Menghitung variabel terakhir x[n-1] (atau x3)
x[n - 1] = M[n - 1, n] / M[n - 1, n - 1]
print(f"Langkah 1: Menghitung x[{n}] (variabel terakhir):")
print(f"x[{n}] = {M[n-1, n]:.6f} / {M[n-1, n-1]:.6f} = {x[n-1]:.6f}\n")

# 2. Loop mundur dari indeks baris n-2 sampai 0 (x2 lalu x1)
for i in range(n - 2, -1, -1):
    # Hitung jumlahan perkalian koefisien dengan variabel yang sudah diketahui sebelumnya
    jumlah_ax = np.dot(M[i, i + 1:n], x[i + 1:n])

    # Hitung nilai x[i] dengan mengurangkan konstanta b' terhadap jumlah_ax lalu dibagi pivot diagonal
    x[i] = (M[i, n] - jumlah_ax) / M[i, i]

    print(f"Langkah {n - i}: Menghitung x[{i + 1}]:")
    print(f"x[{i + 1}] = ({M[i, n]:.6f} - {jumlah_ax:.6f}) / {M[i, i]:.6f} = {x[i]:.6f}\n")

# Menampilkan ringkasan vektor solusi yang diperoleh
print("===========================================")
print("HASIL VEKTOR SOLUSI (METODE ELIMINASI GAUSS)")
print("===========================================")
for idx, val in enumerate(x, start=1):
    print(f"x_{idx} = {val:.8f}")

=== PROSES SUBSTITUSI MUNDUR (BACK SUBSTITUTION) ===

Langkah 1: Menghitung x[3] (variabel terakhir):
x[3] = 10.000000 / 7.500000 = 1.333333

Langkah 2: Menghitung x[2]:
x[2] = (14.500000 - 2.666667) / 4.500000 = 2.629630

Langkah 3: Menghitung x[1]:
x[1] = (9.000000 - -0.037037) / 4.000000 = 2.259259

HASIL VEKTOR SOLUSI (METODE ELIMINASI GAUSS)
x_1 = 2.25925926
x_2 = 2.62962963
x_3 = 1.33333333


### Cell 6: Verifikasi Hasil & Perhitungan Residu Galat
Untuk memastikan kebenaran numerik, kita bandingkan solusi algoritma kita dengan solver bawaan `np.linalg.solve` serta menghitung norma residu $||A\mathbf{x} - \mathbf{b}||_2$.

In [ ]:
# Menghitung solusi pembanding menggunakan pustaka standar NumPy
x_numpy = np.linalg.solve(A, b)

# Menghitung vektor residu r = A*x - b
vektor_residu = np.dot(A, x) - b

# Menghitung norma Euclidean (L2-norm) dari vektor residu
norma_residu = np.linalg.norm(vektor_residu, 2)

# Menghitung selisih absolut antara solusi algoritma kita dengan numpy solver
selisih_maksimal = np.max(np.abs(x - x_numpy))

print("=== VERIFIKASI & ANALISIS GALAT KETELITIAN ===")
print(f"Solusi Algoritma Mandiri : x1 = {x[0]:.8f}, x2 = {x[1]:.8f}, x3 = {x[2]:.8f}")
print(f"Solusi NumPy Benchmark   : x1 = {x_numpy[0]:.8f}, x2 = {x_numpy[1]:.8f}, x3 = {x_numpy[2]:.8f}")
print(f"Selisih Maksimal         : {selisih_maksimal:.2e}")
print(f"Norma Residu ||Ax - b||  : {norma_residu:.2e}")

# Pernyataan evaluasi validitas
if norma_residu < 1e-10:
    print("\n[KESIMPULAN]: Solusi sangat akurat dan terverifikasi secara presisi tinggi!")
else:
    print("\n[KESIMPULAN]: Perlu evaluasi kondisi angka matriks (conditioning number).")

=== VERIFIKASI & ANALISIS GALAT KETELITIAN ===
Solusi Algoritma Mandiri : x1 = 2.25925926, x2 = 2.62962963, x3 = 1.33333333
Solusi NumPy Benchmark   : x1 = 2.25925926, x2 = 2.62962963, x3 = 1.33333333
Selisih Maksimal         : 0.00e+00
Norma Residu ||Ax - b||  : 8.88e-16

[KESIMPULAN]: Solusi sangat akurat dan terverifikasi secara presisi tinggi!


### Cell 7: Fungsi Serbaguna Terintegrasi (Modular Function)
Pada cell ini disajikan fungsi lengkap `selesaikan_spl_gauss_3v(A_input, b_input)` yang modular, dilengkapi penanganan kesalahan (*error handling*), dan dapat menerima sembarang input SPL 3 variabel dari pengguna.

In [ ]:
def selesaikan_spl_gauss_3v(A_mat, b_vec):
    """
    Fungsi penyelesai Sistem Persamaan Linear 3 Variabel (SPL 3V)
    menggunakan metode Eliminasi Gauss dengan Partial Pivoting.

    Parameter:
        A_mat : array-like 3x3 (Matriks Koefisien)
        b_vec : array-like 3 (Vektor Konstanta)

    Return:
        solusi : numpy array 1D berisi [x1, x2, x3]
    """
    # Konversi input menjadi array float64
    A_arr = np.array(A_mat, dtype=np.float64)
    b_arr = np.array(b_vec, dtype=np.float64)

    # Validasi dimensi input
    if A_arr.shape != (3, 3):
        raise ValueError("Matriks A harus berukuran tepat 3x3!")
    if b_arr.shape != (3,):
        raise ValueError("Vektor b harus memiliki tepat 3 elemen!")

    # Pembentukan matriks teraugmentasi
    n_dim = 3
    aug = np.hstack([A_arr, b_arr.reshape(-1, 1)])

    # Eliminasi Maju dengan Partial Pivoting
    for k_col in range(n_dim - 1):
        # Cari indeks pivot maksimum
        p_max = k_col + np.argmax(np.abs(aug[k_col:, k_col]))
        if np.abs(aug[p_max, k_col]) < 1e-12:
            raise ValueError("SPL tidak memiliki solusi unik (Matriks Singular).")

        # Tukar baris jika diperlukan
        if p_max != k_col:
            aug[[k_col, p_max]] = aug[[p_max, k_col]]

        # Eliminasi baris di bawahnya
        for i_row in range(k_col + 1, n_dim):
            m_factor = aug[i_row, k_col] / aug[k_col, k_col]
            aug[i_row, k_col:] -= m_factor * aug[k_col, k_col:]

    # Cek pivot diagonal terakhir
    if np.abs(aug[n_dim - 1, n_dim - 1]) < 1e-12:
        raise ValueError("Matriks singular pada elemen pivot terakhir.")

    # Substitusi Mundur
    sol = np.zeros(n_dim, dtype=np.float64)
    sol[n_dim - 1] = aug[n_dim - 1, n_dim] / aug[n_dim - 1, n_dim - 1]
    for i_row in range(n_dim - 2, -1, -1):
        dot_sum = np.dot(aug[i_row, i_row + 1:n_dim], sol[i_row + 1:n_dim])
        sol[i_row] = (aug[i_row, n_dim] - dot_sum) / aug[i_row, i_row]

    return sol

# Contoh Pengujian dengan Kasus Baru yang memiliki Solusi Bulat Elegan
# SPL Baru:
#  x +  y +  z = 6
# 2x -  y +  z = 3
# 3x + 2y -  z = 4
# (Solusi eksak: x = 1, y = 2, z = 3)

A_uji = [[1, 1, 1], [2, -1, 1], [3, 2, -1]]
b_uji = [6, 3, 4]

hasil_uji = selesaikan_spl_gauss_3v(A_uji, b_uji)
print("=== Uji Coba Fungsi Modular pada Kasus SPL Baru ===")
print(f"Solusi diperoleh: x = {hasil_uji[0]:.4f}, y = {hasil_uji[1]:.4f}, z = {hasil_uji[2]:.4f}")

=== Uji Coba Fungsi Modular pada Kasus SPL Baru ===
Solusi diperoleh: x = 1.0000, y = 2.0000, z = 3.0000
